# forget-it — results analysis

Three analyses over `combined_results.csv`, each answering a different question:

1. **Pressure diagnostic** — which configs actually evicted? (Most didn't.)
2. **Policy ranking** — at the budget where eviction bites, which forgetting policy wins?
3. **Reading-loss decomposition** — where does the answer actually get lost: retention, retrieval, or reading?

Section 4 sketches the **survival / regret / value-decay** curves; those need a per-eviction
event log that current runs don't emit yet, so that section is guarded and no-ops until the log exists.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

# robust whether you run from repo root or scripts/
candidates = [
    Path("artifacts/10-05-results/combined_results.csv"),
    Path("../artifacts/10-05-results/combined_results.csv"),
]
RESULTS = next((p for p in candidates if p.exists()), candidates[0])
df = pd.read_csv(RESULTS)
print(f"loaded {len(df):,} scored-query rows from {RESULTS}")
print("policies:", sorted(df.policy.unique()), "| budgets:", sorted(df.budget_bytes.unique()))
df.head(3)

loaded 35,500 scored-query rows from ../artifacts/10-05-results/combined_results.csv
policies: ['compression', 'lfu', 'lru', 'random'] | budgets: [np.int64(262144), np.int64(524288), np.int64(1048576), np.int64(2097152), np.int64(5242880)]


,dataset,query_id,policy,budget_bytes,itemiser,retriever,reader,seed,k,gold_survived,...,latency_sec,compressions,reader_answer,abstained,qa_latency_sec,ref_answer,question_text,question_type,store,introspectable
0,longmemeval,e47becba,compression,1048576,session,bm25,PlainTextReader,42,5,True,...,0.026147,0,IDK,True,0.098544,Business Administration,What degree did I graduate with?,single-session-user,retention/compression/bm25/budget_1048576,True
1,longmemeval,118b2229,compression,1048576,session,bm25,PlainTextReader,42,5,True,...,0.027340,0,IDK,True,0.113233,45 minutes each way,How long is my daily commute to work?,single-session-user,retention/compression/bm25/budget_1048576,True
2,longmemeval,51a45a95,compression,1048576,session,bm25,PlainTextReader,42,5,True,...,0.044527,0,IDK,True,0.098812,Target,Where did I redeem a $5 coupon on coffee creamer?,single-session-user,retention/compression/bm25/budget_1048576,True


## 1. Pressure diagnostic — which configs actually forgot?

Eviction/compression only happens when the haystack overflows the budget. If a config
never evicted, every policy trivially scores ~100% there and the comparison is vacuous.

In [2]:
pressure = (df.groupby(["policy", "budget_bytes"])
              [["evictions_total_at_query_time", "compressions"]]
              .mean().round(1))
pressure

evictions_total_at_query_time  compressions
policy      budget_bytes                                             
compression 524288                                  0.0           0.0
            1048576                                 0.0           0.0
            2097152                                 0.0           0.0
            5242880                                 0.0           0.0
lfu         262144                              18199.6           0.0
            524288                                  0.0           0.0
            1048576                                 0.0           0.0
            2097152                                 0.0           0.0
            5242880                                 0.0           0.0
lru         262144                              17847.9           0.0
            524288                                  0.0           0.0
            1048576                                 0.0           0.0
            2097152                                 0.0           0.0
            5242880                                 0.0           0.0
random      262144                              17521.6           0.0
            524288                                  0.0           0.0
            1048576                                 0.0           0.0
            2097152                                 0.0           0.0
            5242880                                 0.0           0.0

In [ ]:
# --- Diagnostic: is `evictions_total_at_query_time` per-example or cumulative? ---
# It reads logger.counts["evict"], which only increments and is NOT reset by clear(),
# so it is a CUMULATIVE run-wide odometer — easy to misread as "evictions per example."
# Proof + the REAL per-example count (the delta between consecutive queries):
g = (df[(df.policy == "lfu") & (df.budget_bytes == 262144)
        & (df.itemiser == "session") & (df.retriever == "bm25")]
       .reset_index(drop=True))
e = g.evictions_total_at_query_time
delta = e.diff()  # evictions that happened during each example

print(f"queries in this config: {len(g)}")
print(f"evictions_total: first={e.iloc[0]}, last={e.iloc[-1]}, mean={e.mean():.0f}")
print(f"monotonic non-decreasing across queries? {e.is_monotonic_increasing}  "
      f"=> CUMULATIVE across the run, NOT per-example")
print(f"actual per-example evictions (delta): median={delta.median():.0f}, max={delta.max():.0f}")
print("A session example has ~50 items, so ~25 evictions/example < items present")
print("=> insert-once / evict-once holds. No thrash, no double-count.")
print("The cumulative count is a churn odometer, not a damage gauge — use the section-4 "
      "survival/regret/value-decay curves for damage.")

Read this before trusting any average: budgets with `evictions = 0` are no-ops.
In this run only the smallest budget applies pressure, and the compression policy
logged **0 compressions** everywhere (and is missing at the pressured budget) — so
compression is **not yet evaluable** from this data.

## 2. Policy ranking — only where eviction bites, and only WITHIN a cell

Keep only rows where forgetting actually happened. Rank policies by **`gold_survived`**
(the policy's own verdict). Note `hit_at_k_given_retained` *conditions out* eviction, so
it is policy-insensitive by design — use it to judge the retriever, **not** to rank policies.

**Ranking is per `(itemiser, retriever)` cell, never pooled** (R-4). The itemiser sets item
granularity — it decides what a byte budget even holds — and the retriever sets the ranking
that produces every `hit_at_k`. Averaging a policy over both mixes populations whose
denominators differ, and the pooled mean is then dominated by whichever cell happens to
contribute the most rows. The pooled table is kept below as a diagnostic, clearly labelled,
with a check for whether pooling actually reverses any within-cell ordering.

In [ ]:
P = df[df.evictions_total_at_query_time > 0]
print(f"pressured rows: {len(P):,} | policies: {sorted(P.policy.unique())} "
      f"| budgets: {sorted(P.budget_bytes.unique())}")

metrics = ["gold_survived", "hit_at_k", "hit_at_k_given_retained", "mrr"]

# R-4: rank on the COUPLED key. `itemiser` changes what an item (and therefore a
# byte budget) is; `retriever` changes the ranking every hit_at_k is read off.
# Pooling over them averages populations with different denominators, so the
# result tracks row counts as much as policy quality.
CELL = ["itemiser", "retriever"]
ranked = (P.groupby(CELL + ["policy"])[metrics].mean().round(3)
            .sort_values(CELL + ["gold_survived"], ascending=[True, True, False]))
ranked

In [ ]:
# DIAGNOSTIC ONLY — the POOLED table. Not a ranking (R-4): it averages over
# itemiser x retriever. Kept so the pooled numbers stay inspectable, and so the
# check below can say whether pooling would have changed any conclusion.
pooled = P.groupby("policy")[metrics].mean().round(3).sort_values("gold_survived", ascending=False)
print("POOLED over itemiser x retriever — DIAGNOSTIC, NOT A RANKING:")
display(pooled)

# Does pooling reverse any within-cell ordering? If yes, the pooled table is not
# merely coarse, it is wrong, and any claim read off it has to be withdrawn.
pooled_order = list(pooled.index)
disagreements = []
for cell, g in P.groupby(CELL):
    cell_order = list(g.groupby("policy")["gold_survived"].mean()
                       .sort_values(ascending=False).index)
    if cell_order != [p for p in pooled_order if p in cell_order]:
        disagreements.append((cell, cell_order))

print(f"\npooled order: {pooled_order}")
if disagreements:
    print(f"POOLING REVERSES {len(disagreements)} of {P.groupby(CELL).ngroups} cells:")
    for cell, order in disagreements:
        print(f"  {cell}: {order}")
else:
    print(f"no reversals across {P.groupby(CELL).ngroups} cells — pooling is coarse here, "
          "not wrong. This is a property of THIS data, not a licence to pool.")

**Takeaway (read off the per-cell table, not the pooled one):** LRU ≈ LFU, and both clearly
beat Random on `gold_survived` and `hit_at_k` **in every `(itemiser, retriever)` cell** — the
probe fix works on real data. `hit_at_k_given_retained` is ~flat across all three (including
Random), confirming it measures retrieval, not the policy.

The claim is stated per cell because that is where it holds; the pooled diagnostic above
happens to agree here, which is a fact about this dataset and not a reason to pool the next one.

## 3. Reading-loss decomposition — retention → retrieval → reading

A query is answerable only if the gold item (1) survived eviction, (2) was retrieved in
the top-k, and (3) the reader actually answers from it. **Reading loss** = gold survived
*and* retrieved at top (`gold_survived & hit_at_k == 1`) but the reader **abstained**.

In [5]:
survived     = df.gold_survived.astype(bool)
retrieved    = survived & (df.hit_at_k == 1.0)
answered     = retrieved & ~df.abstained.astype(bool)
reading_loss = retrieved &  df.abstained.astype(bool)
n = len(df)

print(f"retrievable (gold survived & hit@k==1): {retrieved.sum():,}  ({retrieved.mean():.1%})")
print(f"reading-loss (retrievable & abstained): {reading_loss.sum():,}  ({reading_loss.mean():.1%} of all)")
print(f"  -> when retrievable, the reader abstained {reading_loss.sum()/retrieved.sum():.1%} of the time")
print(f"overall abstention rate: {df.abstained.mean():.1%}")
print(f"  -> of all abstentions, {reading_loss.sum()/df.abstained.sum():.1%} had the answer retrievable")

retrievable (gold survived & hit@k==1): 30,536  (86.0%)
reading-loss (retrievable & abstained): 20,531  (57.8% of all)
  -> when retrievable, the reader abstained 67.2% of the time
overall abstention rate: 70.9%
  -> of all abstentions, 81.6% had the answer retrievable


### Loss funnel (every query partitioned into exactly one bucket)

In [6]:
funnel = pd.Series({
    "total":                                n,
    "retention loss (gold evicted)":        (~survived).sum(),
    "retrieval loss (kept, not in top-k)":  (survived & (df.hit_at_k != 1.0)).sum(),
    "reading loss (retrieved, abstained)":  reading_loss.sum(),
    "answered (retrieved, committed)":      answered.sum(),
})
pd.DataFrame({"rows": funnel, "pct_of_total": (funnel / n * 100).round(1)})

,rows,pct_of_total
total,35500,100.0
retention loss (gold evicted),625,1.8
"retrieval loss (kept, not in top-k)",4339,12.2
"reading loss (retrieved, abstained)",20531,57.8
"answered (retrieved, committed)",10005,28.2


### Reading loss by question type

If abstention were *wise refusal* it would cluster on hard types. If it's the reader
discarding answers it holds, it stays high even on concrete, answerable types.

In [7]:
by_qt = pd.DataFrame({
    "retrievable":  retrieved.groupby(df.question_type).sum(),
    "reading_loss": reading_loss.groupby(df.question_type).sum(),
})
by_qt["abstain_despite_retrieval"] = (by_qt.reading_loss / by_qt.retrievable).round(3)
by_qt.sort_values("abstain_despite_retrieval", ascending=False)

,retrievable,reading_loss,abstain_despite_retrieval
question_type,,,
single-session-assistant,3329,2833,0.851
temporal-reasoning,8263,6342,0.768
multi-session,8212,5654,0.689
single-session-preference,1069,681,0.637
single-session-user,4409,2622,0.595
knowledge-update,5254,2399,0.457


### Is it the policy or the reader?

Reading loss happens *after* retention/retrieval, so it should be ~constant across
eviction policies — i.e. a reader problem, not a policy problem.

In [8]:
pd.DataFrame({
    "retrievable":       retrieved.groupby(df.policy).sum(),
    "reading_loss_rate": (reading_loss.groupby(df.policy).sum()
                          / retrieved.groupby(df.policy).sum()),
}).round(3)

,retrievable,reading_loss_rate
policy,,
compression,6517,0.669
lfu,8305,0.666
lru,8252,0.668
random,7462,0.687


### Rough correctness proxy (caveated)

The pipeline does **not** grade answers. Below is a rough "reference appears in answer"
check over committed (non-abstained) answers only — indicative, not a metric (free-text
refs miss paraphrases and over-credit loose matches).

In [9]:
def _norm(s):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", str(s).lower()).split())

committed = df[answered].copy()
committed["rough_correct"] = [
    bool(_norm(r)) and _norm(r) in _norm(a)
    for a, r in zip(committed.reader_answer, committed.ref_answer)
]
print(f"committed answers (retrievable & not abstained): {len(committed):,}")
print(f"  rough correctness (ref substring in answer):   {committed.rough_correct.mean():.1%}")
print("note: rough proxy only -- free-text refs; not a headline score.")

committed answers (retrievable & not abstained): 10,005
  rough correctness (ref substring in answer):   44.6%
note: rough proxy only -- free-text refs; not a headline score.


**Takeaway:** retention+retrieval lose little; the dominant loss is **reading** — the
reader abstains on most queries whose answer it already holds, and is right less than half
the time when it does commit. The headline abstention rate is mostly *discarded answers*,
not refusal, and it is independent of the eviction policy.

## 4. Survival / regret / value-decay curves (needs a per-eviction log)

These turn binary `gold_survived` into continuous, time-resolved scores. They require a
per-run event trace the current runs don't emit yet — `events.jsonl` with rows like
`{"name":"insert"|"evict"|"clear","id":...,"t_insert":<insert ordinal>}` plus a
`{"name":"gold","gold_ids":[...]}` marker per query. To produce it: give the per-config
`BenchmarkLogger` a path, add `t_insert` to insert/evict payloads in `RetentionManager`,
and log the gold ids in the runner.

The functions below compute, once that log exists:
- **(1) Kaplan–Meier** gold-survival curve + AUC (restricted mean survival time),
- **(2) per-eviction Belady regret** (evicted gold while a non-gold was resident),
- **(3) value-decay** `V(t)` = #gold resident over time, and its integral
  (`∫V` equals the sum of gold lifetimes — the dual of the KM AUC).

In [ ]:
def survival_regret_decay(events_path):
    """KM RMST, Belady regret, and value-decay integral from one events.jsonl.

    Handles clear_between_examples: an item still resident at a clear is CENSORED
    at that clear (it survived its example), not at the global end of the run.
    KM RMST needs `lifelines` (pip install lifelines); regret and value-decay don't.
    """
    ev = pd.read_json(events_path, lines=True)
    ev = ev.sort_values("seq" if "seq" in ev.columns else "t").reset_index(drop=True)
    # ids -> plain python str (pyarrow string scalars don't hash-match str in sets)
    ev["id"] = ev["id"].map(lambda v: None if pd.isna(v) else str(v))

    gold = set()
    if "gold_ids" in ev.columns:
        for x in ev.loc[ev.name == "gold", "gold_ids"].dropna():
            gold |= {str(i) for i in x}

    clears = sorted(ev.loc[ev.name == "clear", "t_insert"].tolist())
    last = ev["t_insert"].max()
    def next_clear_after(t):
        return next((c for c in clears if c >= t), last)   # censor at next clear, else end

    ins = ev[ev.name == "insert"][["id", "t_insert"]].rename(columns={"t_insert": "born"})
    evi = ev[ev.name == "evict"][["id", "t_insert"]].rename(columns={"t_insert": "died"})
    evicted_ids = set(evi.id)

    life = ins.merge(evi, on="id", how="left")
    life["evicted"] = life.id.isin(evicted_ids)
    life["exit"]    = [d if (i in evicted_ids and pd.notna(d)) else next_clear_after(b)
                       for i, b, d in zip(life.id, life.born, life.died)]
    life["lifetime"] = (life["exit"] - life["born"]).clip(lower=0)
    life["is_gold"]  = life.id.isin(gold)
    g = life[life.is_gold]

    # (1) Kaplan-Meier restricted mean survival time (optional dependency)
    km, rmst = None, None
    try:
        from lifelines import KaplanMeierFitter
        km = KaplanMeierFitter().fit(g.lifetime, event_observed=g.evicted)
        sf = km.survival_function_["KM_estimate"]
        rmst = float((sf.values[:-1] * np.diff(sf.index.values)).sum())
    except ImportError:
        pass  # lifelines not installed; regret + value-decay below still work

    # (2) Belady regret + (3) value-decay, replaying insert/evict/clear in order.
    # NB: use e["name"] — a Series' .name is its index label, NOT the "name" column.
    resident, regret = set(), 0
    V_prev, t_prev, V_area = 0, None, 0.0
    for _, e in ev[ev.name.isin(["insert", "evict", "clear"])].iterrows():
        kind, rid, t = e["name"], e["id"], e["t_insert"]
        if t_prev is not None:                      # accumulate area under V(t)
            V_area += V_prev * (t - t_prev)
        if kind == "insert":
            resident.add(rid)
        elif kind == "clear":
            resident.clear()
        else:                                       # evict
            if rid in gold and any(r not in gold for r in resident):
                regret += 1                         # avoidable: killed gold, non-gold available
            resident.discard(rid)
        V_prev = sum(1 for r in resident if r in gold)
        t_prev = t

    # mean gold lifetime is a dependency-free survival summary; value_decay_auc
    # (= area under #gold-resident) is its dual.
    return {"n_gold": int(len(g)),
            "mean_gold_lifetime": float(g.lifetime.mean()) if len(g) else 0.0,
            "km_rmst": rmst, "belady_regret": regret,
            "value_decay_auc": float(V_area), "km": km}


logs = sorted(Path("artifacts").rglob("events.jsonl"))
if not logs:
    print("No events.jsonl found yet. Re-run a pressured config with --event-log "
          "(e.g. run_longmemeval_qa --event-log) to populate these curves.")
else:
    for p in logs:
        r = {k: v for k, v in survival_regret_decay(p).items() if k != "km"}
        print(p.parent.name, r)